# 05. Human-in-the-Loop Approval Workflow

[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue.svg)](https://www.python.org/)
[![LangGraph](https://img.shields.io/badge/LangGraph-1.2.11%2B-orange.svg)](https://github.com/langchain-ai/langgraph)
[![LangChain](https://img.shields.io/badge/LangChain-1.3%2B-green.svg)](https://github.com/langchain-ai/langchain)
[![Google Gemini](https://img.shields.io/badge/Model-Gemini%202.5%20Flash-4285F4.svg)](https://aistudio.google.com/)

This project demonstrates the **Human-in-the-Loop (HITL)** pattern in **LangGraph**, where an AI workflow pauses execution, waits for human review and feedback, preserves its execution state using checkpointing, and resumes dynamically based on human decisions (Approve vs Reject).

## 🏛️ Architecture Flow

```text
                              START
                                │
                                ▼
                        Generate Content
                                │
                                ▼
                      Human Review (PAUSE)
                                │
                    ┌───────────┴───────────┐
                    │                       │
                    ▼                       ▼
                 Approve                  Reject
                    │                       │
                    ▼                       ▼
              Finalize Content       Improve Content
                    │                       │
                    ▼                       │
                   END                      │
                                            ▼
                                   Human Review (PAUSE)
```

### Key Concepts Introduced:
1. **Workflow Interruption (`interrupt`)**: Halting graph execution at a specific node while awaiting external input.
2. **State Checkpointing (`MemorySaver`)**: Persisting graph state and memory across pause/resume cycles.
3. **Resumption (`Command(resume=...)`)**: Passing human feedback back to the paused node to resume graph execution seamlessly.
4. **Iterative Refinement**: Loop back mechanism allowing multiple rounds of human critique and AI improvement.

## 📦 1. Installation & Environment Setup

Ensure dependencies are installed and load your `GOOGLE_API_KEY` from the `.env` file.

In [ ]:
# Optional: Install required dependencies
# !pip install langgraph langchain langchain-google-genai python-dotenv rich ipywidgets grandalf

In [ ]:
import os
import uuid
from typing import TypedDict, Optional, List, Dict, Any
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage

# Load environment variables
load_dotenv()

google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.6-flash')

print(f"API Key Configured: {bool(google_api_key and not google_api_key.startswith('your_'))}")
print(f"Gemini Model: {gemini_model}")

## 🧠 2. LLM Initialization with Fallback Support

We set up a helper function to initialize `ChatGoogleGenerativeAI`, with fallback support if no live key is configured.

In [ ]:
def get_llm():
    api_key = os.getenv('GOOGLE_API_KEY')
    model_name = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')
    if not api_key or api_key.startswith('your_') or 'invalid' in api_key.lower():
        return None
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=api_key,
            temperature=0.7
        )
    except Exception:
        return None

def generate_llm_draft(topic: str) -> str:
    llm = get_llm()
    if llm:
        try:
            prompt = (
                f"You are a professional content creator. Write an engaging, well-structured "
                f"post or brief article about the following topic:\n\n"
                f"Topic: {topic}\n\n"
                f"Guidelines:\n"
                f"- Include a compelling hook\n"
                f"- 2-3 concise body paragraphs with actionable insights\n"
                f"- A strong takeaway or call-to-action\n"
                f"- Use clear formatting and bullet points."
            )
            response = llm.invoke([HumanMessage(content=prompt)])
            return response.content.strip()
        except Exception as e:
            print(f"Note: Gemini API fallback triggered ({e}).")
            
    return (
        f"🚀 The Future of {topic}: Why It Matters Today\n\n"
        f"In today's fast-evolving technological landscape, {topic} is rapidly shifting from an experimental concept "
        f"into an essential driver of innovation. Organizations that embrace this paradigm are seeing "
        f"unprecedented gains in efficiency, decision velocity, and creative problem-solving.\n\n"
        f"Key Highlights:\n"
        f"• Autonomous Execution: Moving beyond simple static queries to goal-oriented multi-step reasoning.\n"
        f"• Human-AI Synergy: Pairing automated intelligence with human oversight for safety and precision.\n"
        f"• Scalable Reliability: Structuring workflows with robust state management and error resilience.\n\n"
        f"As we look ahead, the defining advantage won't just be having AI capabilities, but orchestrating them seamlessly "
        f"with human expertise.\n\n"
        f"💡 What are your thoughts on how {topic} will shape your industry? Share below!"
    )

def refine_llm_draft(topic: str, previous_draft: str, feedback: str, iteration: int) -> str:
    llm = get_llm()
    if llm:
        try:
            prompt = (
                f"You are an expert editor refining content based on human reviewer feedback.\n\n"
                f"Original Topic: {topic}\n\n"
                f"Previous Draft (Iteration {iteration - 1}):\n"
                f"{previous_draft}\n\n"
                f"Human Reviewer Feedback:\n"
                f"{feedback}\n\n"
                f"Task:\n"
                f"Rewrite and improve the draft strictly addressing all feedback while maintaining high quality."
            )
            response = llm.invoke([HumanMessage(content=prompt)])
            return response.content.strip()
        except Exception as e:
            print(f"Note: Gemini API fallback triggered ({e}).")
            
    return (
        f"✨ [Refined Iteration {iteration}] {topic}\n\n"
        f"Addressing Reviewer Notes: '{feedback}'\n\n"
        f"In modern enterprise systems, {topic} fundamentally transforms operational workflows. "
        f"Rather than operating in an isolated silo, modern implementations focus on actionable business outcomes, "
        f"lower latency, and verifiable human control.\n\n"
        f"Core Takeaways:\n"
        f"1. Precision & Focus: Directly tailored to meet user feedback ({feedback}).\n"
        f"2. Controlled Automation: Predictable guardrails ensuring compliance and high reliability.\n"
        f"3. Measurable Impact: Streamlined execution delivering tangible ROI.\n\n"
        f"Summary: Continuous refinement ensures both AI speed and human quality standards."
    )

## 📋 3. State Schema Definition

The state is passed between all nodes and recorded by the checkpointer at each step.

In [ ]:
class ContentState(TypedDict):
    topic: str
    generated_content: str
    approval_status: Optional[str]  # 'approved' | 'rejected' | 'pending'
    human_feedback: Optional[str]
    final_content: Optional[str]
    iteration_count: int
    revision_history: List[Dict[str, Any]]

## ⚙️ 4. Graph Nodes & Conditional Routing

Here we define:
- `generate_content`: Generates the initial draft.
- `human_review`: Uses `interrupt()` to pause the graph and return the draft to the caller. When resumed via `Command(resume=...)`, receives the human's decision.
- `decide_next_step`: Conditional router (`approve` -> `finalize_content`, `reject` -> `improve_content`).
- `improve_content`: Incorporates human feedback into a new revised draft.
- `finalize_content`: Prepares the final approved result.

In [ ]:
def generate_content(state: ContentState) -> Dict[str, Any]:
    topic = state.get('topic', 'AI Systems')
    print(f"[Node: generate_content] Generating initial draft for: '{topic}'...")
    draft = generate_llm_draft(topic)
    
    return {
        'generated_content': draft,
        'iteration_count': 1,
        'approval_status': 'pending',
        'revision_history': [
            {
                'iteration': 1,
                'content': draft,
                'action': 'generated',
                'feedback': None
            }
        ]
    }

def human_review(state: ContentState) -> Dict[str, Any]:
    iteration = state.get('iteration_count', 1)
    content = state.get('generated_content', '')
    
    print(f"\n[Node: human_review] ⏸️ PAUSING workflow at iteration {iteration} for Human Review...")
    
    # Interrupt execution and deliver payload to client
    review_response = interrupt({
        'message': 'Please review the generated content and choose Approve or Reject with feedback.',
        'topic': state.get('topic'),
        'iteration': iteration,
        'content': content
    })
    
    print(f"[Node: human_review] ▶️ RESUMED with decision: {review_response}")
    
    if not isinstance(review_response, dict):
        review_response = {'action': 'approve' if str(review_response).lower() in ('approve', 'yes', '1') else 'reject'}
        
    action = review_response.get('action', 'reject').lower()
    feedback = review_response.get('feedback', '')
    
    history = list(state.get('revision_history', []))
    history.append({
        'iteration': iteration,
        'action': action,
        'feedback': feedback
    })
    
    return {
        'approval_status': action,
        'human_feedback': feedback,
        'revision_history': history
    }

def decide_next_step(state: ContentState) -> str:
    status = state.get('approval_status', '').lower()
    if status in ('approve', 'approved', 'accept'):
        return 'finalize_content'
    return 'improve_content'

def improve_content(state: ContentState) -> Dict[str, Any]:
    topic = state.get('topic', '')
    previous_draft = state.get('generated_content', '')
    feedback = state.get('human_feedback', 'Make it more concise.')
    new_iteration = state.get('iteration_count', 1) + 1
    
    print(f"[Node: improve_content] Improving draft based on feedback: '{feedback}' (Iteration {new_iteration})...")
    refined_draft = refine_llm_draft(topic, previous_draft, feedback, new_iteration)
    
    history = list(state.get('revision_history', []))
    history.append({
        'iteration': new_iteration,
        'content': refined_draft,
        'action': 'improved',
        'feedback_addressed': feedback
    })
    
    return {
        'generated_content': refined_draft,
        'iteration_count': new_iteration,
        'approval_status': 'pending',
        'revision_history': history
    }

def finalize_content(state: ContentState) -> Dict[str, Any]:
    print('[Node: finalize_content] ✅ Finalizing approved content...')
    return {
        'final_content': state.get('generated_content', ''),
        'approval_status': 'approved'
    }

## 🔨 5. Graph Compilation with Checkpointing

We compile the graph with a `MemorySaver` checkpointer so that thread state is preserved across interrupts.

In [ ]:
builder = StateGraph(ContentState)

# Add nodes
builder.add_node('generate_content', generate_content)
builder.add_node('human_review', human_review)
builder.add_node('improve_content', improve_content)
builder.add_node('finalize_content', finalize_content)

# Add edges
builder.add_edge(START, 'generate_content')
builder.add_edge('generate_content', 'human_review')

builder.add_conditional_edges(
    'human_review',
    decide_next_step,
    {
        'finalize_content': 'finalize_content',
        'improve_content': 'improve_content'
    }
)

builder.add_edge('improve_content', 'human_review')
builder.add_edge('finalize_content', END)

# Compile with checkpointer
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)
print('Workflow graph compiled successfully with MemorySaver checkpointer!')

## 📊 6. Graph Visualization

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    try:
        print(app.get_graph().draw_ascii())
    except Exception:
        print('Nodes:', list(app.get_graph().nodes.keys()))
        print('Edges:', [(e.source, e.target) for e in app.get_graph().edges])

## 🚀 7. Step-by-Step Execution Walkthrough

### Step 7.1: Initial Run Until Interrupt

In [ ]:
thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

initial_topic = 'Write a professional LinkedIn post about the importance of AI agents.'
print(f"Starting workflow for Thread ID: {thread_id}\n")

# Stream until interrupt is triggered
for event in app.stream({'topic': initial_topic}, config=config):
    print('Stream Event:', event)

### Step 7.2: Inspecting Checkpoint State at Pause

Let's check the current checkpoint state, next tasks, and the interrupt payload.

In [ ]:
current_state = app.get_state(config)

print('--- Checkpoint State ---')
print(f"Next Node to Execute: {current_state.next}")
print(f"Current Iteration: {current_state.values.get('iteration_count')}")
print(f"Generated Content Preview:\n{current_state.values.get('generated_content')}\n")

if current_state.tasks:
    for task in current_state.tasks:
        print('Active Interrupts:', [i.value for i in task.interrupts])

### Step 7.3: Simulating Rejection with Human Feedback

We provide human feedback requesting the post be shorter and focused on enterprise ROI. We resume using `Command(resume=...)`.

In [ ]:
human_decision_1 = {
    'action': 'reject',
    'feedback': 'Make the post shorter, punchier, and emphasize enterprise workflow automation.'
}

print('Resuming graph with Rejection + Feedback...')
for event in app.stream(Command(resume=human_decision_1), config=config):
    print('Stream Event:', event)

### Step 7.4: Inspecting State after AI Refinement (Iteration 2)

In [ ]:
state_iter2 = app.get_state(config)

print('--- Checkpoint State (Iteration 2) ---')
print(f"Next Node: {state_iter2.next}")
print(f"Iteration Count: {state_iter2.values.get('iteration_count')}")
print(f"Refined Content Preview:\n{state_iter2.values.get('generated_content')}")

### Step 7.5: Resuming with Human Approval

The reviewer is now satisfied with the revised draft and issues an approval.

In [ ]:
human_decision_2 = {
    'action': 'approve',
    'feedback': 'Looks excellent, ready to publish.'
}

print('Resuming graph with Approval...')
for event in app.stream(Command(resume=human_decision_2), config=config):
    print('Stream Event:', event)

### Step 7.6: Inspecting Final Workflow Output

In [ ]:
final_state = app.get_state(config)

print('=' * 60)
print('🎉 FINAL APPROVED CONTENT:')
print('=' * 60)
print(final_state.values.get('final_content'))
print('=' * 60)
print(f"Workflow Completed: {final_state.next == ()}")
print(f"Total Iterations: {final_state.values.get('iteration_count')}")
print('\nRevision History:')
for h in final_state.values.get('revision_history', []):
    print(' - ', h)

## 🎮 8. Interactive Review Loop in Jupyter

Run this cell to test the approval workflow interactively with your own custom topics and live feedback!

In [ ]:
def interactive_jupyter_session(custom_topic: str):
    thread_id = f"jupyter-{uuid.uuid4()}"
    cfg = {"configurable": {"thread_id": thread_id}}
    
    print(f"Starting Interactive Session for: '{custom_topic}'")
    list(app.stream({'topic': custom_topic}, config=cfg))
    
    while True:
        s = app.get_state(cfg)
        if not s.next:
            break
            
        draft = s.values.get('generated_content')
        it = s.values.get('iteration_count', 1)
        
        print(f"\n{'='*50}")
        print(f"📝 Human Review (Iteration {it}):")
        print(f"{'='*50}\n{draft}\n{'='*50}")
        
        action_choice = input("Choose action: [1] Approve, [2] Reject with feedback: ").strip()
        
        if action_choice in ('1', 'approve', 'a'):
            print('\nApproving draft...')
            list(app.stream(Command(resume={'action': 'approve', 'feedback': 'Approved in notebook.'}), config=cfg))
            break
        else:
            fb = input("Enter your feedback for revision: ").strip()
            if not fb:
                fb = 'Please make it more engaging.'
            print('\nSubmitting feedback and refining draft...')
            list(app.stream(Command(resume={'action': 'reject', 'feedback': fb}), config=cfg))
            
    end_state = app.get_state(cfg)
    print('\n✨ Completed! Final Output:')
    print(end_state.values.get('final_content'))

# Uncomment the line below to run an interactive session:
# interactive_jupyter_session('Write an email announcement for a new AI-powered product feature.')

## 💡 9. Summary & Key Takeaways

| LangGraph Concept | Purpose in HITL |
| :--- | :--- |
| `interrupt()` | Pauses the node execution and surfaces data to the caller. |
| `Command(resume=...)` | Delivers the human's response to the interrupted node and resumes graph execution. |
| `MemorySaver` / Checkpointing | Stores full graph state between turns, enabling long pauses, server restarts, or async human review. |
| Conditional Routing | Decides whether to finalize or loop back based on human approval status. |
| Iteration Tracking | Keeps track of revision counts and history for auditing and compliance. |